In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DataType,BooleanType
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog_name","ecommers","Catalog Name")
dbutils.widgets.text("storage_account_name",'ecommcistorage05',"storage account name")
dbutils.widgets.text("container","ecom-raw-data","Container Name")


In [0]:
catalog_name=dbutils.widgets.get("catalog_name")
storage_account_name=dbutils.widgets.get("storage_account_name")
container=dbutils.widgets.get("container")

In [0]:
silver_incremental_data_df= spark.readStream.format("delta").option('readChangeFeed',"true").table(f"{catalog_name}.silver.slv_order_items")

In [0]:
df_union =silver_incremental_data_df.filter("_change_type IN ('insert','update_postimage')")

In [0]:
df_union = df_union.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
    )

# 2) Add discount_amount (discount_pct is already numeric, e.g., 21 -> 21%)
df_union = df_union.withColumn(
    "discount_amount",
    F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
)

# 3) Add sale_amount = gross - discount
df_union = df_union.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
)

# add date id
df_union = df_union.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))  # Create date_key

# Coupon flag
#  coupon flag = 1 if coupon_code is not null else 0
df_union = df_union.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
     .otherwise(F.lit(0))
)

print("Streaming transformations applied — columns: gross_amount, discount_amount, sale_amount, date_id, coupon_flag")

In [0]:
from delta.tables import DeltaTable

gold_checkpoint_path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_items/"
print(gold_checkpoint_path)

def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_order_items"
    # Drop CDF metadata columns — they conflict with enabling CDF on the gold table
    microBatchDF = microBatchDF.drop("_change_type", "_commit_version", "_commit_timestamp")
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        batch_cols = microBatchDF.columns
        col_map = {c: f"batch_table.{c}" for c in batch_cols}
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.order_id = batch_table.order_id AND gold_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdate(set=col_map).whenNotMatchedInsert(values=col_map).execute()

df_union.writeStream \
    .foreachBatch(upsert_to_gold) \
    .option("checkpointLocation", gold_checkpoint_path) \
    .trigger(availableNow=True) \
    .start() \
    .awaitTermination()

In [0]:
spark.sql(f"SELECT count(*) FROM {catalog_name}.gold.gld_fact_order_items").show()

In [0]:
spark.sql(f"SELECT max(dt) FROM {catalog_name}.gold.gld_fact_order_items").show()